# 课后练习解答（05.03_data_exploration）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** dataset.map(func, remove_columns=dataset.column_names) 执行后，数据集字段为？
A. 仅 func 返回的字段
B. 保留原字段
C. 原字段与返回字段共存
D. 空

**解答：** A

**解析：** remove_columns 删除所有原列，只保留 func 返回的新列。


### 问题2（单选题）

**题目：** 对话数据某个 turn 缺少 system 字段，最合理的处理是？
A. 跳过 system 拼接，保留 user/assistant
B. 终止整个样本
C. 用空字符串占位
D. 随机生成 system

**解答：** A

**解析：** system 是可选上下文，缺失时不应丢弃完整对话，也不应伪造内容。


### 问题3（多选题）

**题目：** SFT 文本模板中应包含？
A. system 指令
B. user 提问
C. assistant 回答
D. 结束符/分隔符

**解答：** ABCD

**解析：** 完整指令样本需要角色标记、内容与结束符，模型才能学会格式与停止位置。


### 问题4（多选题）

**题目：** train_test_split(test_size=0.1, seed=42) 的正确理解包括？
A. 同一 seed 可复现切分
B. 训练/验证样本不重叠
C. 能消除类别不平衡
D. 样本顺序被确定

**解答：** ABD

**解析：** 随机切分不保证类别均衡，需要额外分层策略。


### 问题5（判断题）

**题目：** datasets 的 Dataset.map 默认按样本逐一处理，batched 参数默认为 False。

**解答：** 对

**解析：** HuggingFace datasets 的 map 默认 batched=False；需要批量处理时显式设置 batched=True。


### 问题6（判断题）

**题目：** SFT 训练样本以 user 提问结尾，不包含 assistant 回答。

**解答：** 错

**解析：** 监督微调需要 assistant 回答作为标签，样本必须以完整 assistant 回答结尾。


### 问题7（填空题）

**题目：** 格式化对话时，角色开始标记通常为 ____、____、____。

**解答：** <|system|>、<|user|>、<|assistant|>


### 问题8（填空题）

**题目：** 为保证 train/val 切分可复现，train_test_split 应设置 ____ 参数。

**解答：** seed（random_state）


### 问题9（简答题）

**题目：** 为什么拼接后的 text 需要 strip() 并在末尾添加结束标记？

**解答：** strip() 去除首尾多余换行与空格，避免样本以空白开头或结尾；结束标记告诉模型回答边界，训练时学习何时停止生成。


### 问题10（简答题）

**题目：** 如何检测训练/验证集之间的样本泄漏？

**解答：** 对 text 字段做规范化（去空白/统一标点）后，用哈希集合检查训练与验证样本的重叠；对多轮对话还要检测前缀或子串级重复，避免同一对话内容被切分到两边。


### 问题11（代码设计题）

**题目：** 编写 prepare_dataset(json_path, seed=42)，完成 JSON 读取、对话格式化、9:1 切分并返回 train/val。

**解答：** ```python
from datasets import Dataset
from sklearn.model_selection import train_test_split

def format_conversation(conv):
    parts = []
    if conv.get("system"):
        parts.append("<|system|>\n" + conv["system"])
    for turn in conv["conversation"]:
        role = turn["role"]
        content = turn["content"].strip()
        parts.append(f"<|{role}|>\n{content}")
    return "\n".join(parts) + "\n<|end|>"

def prepare_dataset(json_path, seed=42):
    samples = load_json(json_path)
    data = Dataset.from_list([{"text": format_conversation(s)} for s in samples])
    train, val = data.train_test_split(test_size=0.1, seed=seed).values()
    return train, val
```


### 问题12（单选题）

**题目：** 验证 loss 明显高于训练 loss 且差距随训练扩大，最可能是？
A. 过拟合
B. 欠拟合
C. 学习率过低
D. 数据全部重复

**解答：** A

**解析：** 训练拟合增强而验证泛化变差，是过拟合的典型曲线。


### 问题13（多选题）

**题目：** 数据质量检查应覆盖？
A. 空文本/缺失角色
B. 长度异常
C. 训练验证重复样本
D. 标签/角色顺序错误

**解答：** ABCD

**解析：** 四类问题都会污染训练信号，应建立自动化检查流水线。


### 问题14（判断题）

**题目：** 多轮对话样本应保留从 system 到最终 assistant 的完整轮次。

**解答：** 对

**解析：** 截断中间轮次会破坏上下文依赖，无法学习多轮一致性。


### 问题15（简答题）

**题目：** 数据集仅 1596 条时，为什么 9:1 切分的验证集波动大？如何缓解？

**解答：** 验证集只有约 160 条，单次切分的随机波动会显著影响指标；可通过固定 seed、分层切分、多次切分取平均、扩大验证比例或使用交叉验证来降低评估方差。
